### Application 1: Aspect-based sentiment analyser
Performs Aspect Sentiment Classification (ASC) where aspects within a comment are classified into positive, neutral or negative. The output shows the top 3 most positively rated aspects and top 3 most negatively rated aspects, accompanied with their average sentiment score, where -1 is very negative and +1 is very positive.The output also displays the top 3 most frequently mentioned aspects and frequency of their occurency.

This program is executed on google colab using a T4 GPU. To run, upload the `aspect_sentiment_classifier` model and this notebook, `aspect_based_sentiment_analyser`, to google colab. Furthermore, replace the API_KEY placeholder with a real OpenAI API key.

In [1]:
!pip uninstall -y unsloth unsloth_zoo
!pip -q install -U pip setuptools wheel
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

from openai import OpenAI
import ast
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 37.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 64.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 55.0 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    

In [ ]:
# sample feedback
feedback_list = [
    "The assignments were well-structured and really helped reinforce the concepts taught in lectures.",
    "I felt that the assignments were too time-consuming compared to their contribution to the final grade.",
    "The weekly assignments kept me consistent with the module content and prevented last-minute cramming.",
    "Some assignments felt repetitive and did not add much new learning beyond the tutorials.",
    "The assignments were challenging but fair, and I learned a lot by working through them.",
    "I appreciated how the assignments gradually increased in difficulty throughout the semester.",
    "The assignment instructions were often unclear, which made it stressful to know what was expected.",
    "The assignments aligned closely with the exam format, which made revision much easier.",
    "While the assignments were interesting, the workload was overwhelming when combined with other modules.",
    "The open-ended nature of the assignments encouraged deeper thinking and creativity.",
    "I struggled with the assignments at the start because there were not enough worked examples provided.",
    "The assignments helped me identify my weak areas early in the semester.",
    "Feedback for the assignments was detailed and helped me improve in later submissions.",
    "Some assignments felt disconnected from the lecture content and required a lot of self-learning.",
    "The assignments were practical and gave me hands-on experience applying theoretical concepts.",
    "I found the assignment deadlines too closely spaced, especially during midterm period.",
    "The assignments were engaging and made the module more enjoyable overall.",
    "Grading for the assignments felt inconsistent across different markers.",
    "The assignments pushed me to think critically rather than just apply formulas.",
    "I appreciated that the assignments encouraged collaboration while still being individually assessed.",
    "The difficulty level of the assignments was higher than expected for this module.",
    "The assignments were manageable and well-paced across the semester.",
    "Some assignments felt more like busywork than meaningful learning activities.",
    "The assignments helped bridge the gap between theory and real-world applications.",
    "I often spent more time interpreting the assignment requirements than solving the actual problems.",
    "The assignments were rewarding, especially when I could see my improvement over time.",
    "The scope of the assignments was too broad for the given time frame.",
    "I liked that the assignments allowed multiple approaches rather than a single correct answer.",
    "The assignments were stressful but ultimately prepared me well for the final assessment.",
    "Overall, the classes were a valuable component of the module and enhanced my understanding.",
    "The instructor maintained a good pace throughout the lectures, making it easy to follow along.",
    "I felt that the lectures were too fast, especially when new concepts were introduced.",
    "The pacing of the class was well-balanced and allowed enough time to understand each topic.",
    "Sometimes the instructor moved too quickly through important concepts without sufficient explanation.",
    "The pace was appropriate and kept the lectures engaging without feeling rushed.",
    "I struggled to keep up because the instructor covered too much content in a short time.",
    "The instructor adjusted the pace based on student feedback, which was very helpful.",
    "The lectures occasionally felt slow, particularly during revision of basic concepts.",
    "I appreciated that the instructor paused regularly to check for understanding.",
    "The pace of the lectures was inconsistent, with some sessions feeling rushed and others too slow.",
    "The instructor went through examples at a manageable speed, which helped reinforce learning.",
    "There was not enough time given to absorb complex topics before moving on.",
    "The pacing was ideal for someone with prior knowledge, but difficult for beginners.",
    "I found the lectures too slow at times, which made it hard to stay focused.",
    "The instructor maintained a steady pace that made the content easy to digest.",
    "Key concepts were sometimes rushed, making it difficult to fully understand them.",
    "The instructor took time to explain difficult ideas, which balanced out the overall pace.",
    "The lectures felt too fast, especially when covering technical material.",
    "I liked that the instructor revisited previous topics, which helped manage the pace.",
    "The pacing was too slow for me, as too much time was spent on simple examples.",
    "The instructor managed the pace well, ensuring most students could follow along.",
    "There were moments where the lecture speed increased significantly, making it hard to keep up.",
    "The pace allowed enough time for note-taking and asking questions.",
    "I often felt rushed during lectures and could not fully process the material.",
    "The instructor paced the lessons effectively, balancing theory and examples.",
    "The lectures were sometimes too slow, especially when repeating previously covered material.",
    "The instructor spoke and presented at a pace that was easy to follow.",
    "The pace was too fast when introducing new topics, but slower during revisions.",
    "I appreciated the consistent pacing across all lectures.",
    "Overall, the instructor’s pacing supported my understanding of the module."
]

In [ ]:
# instantiate api connection
client = OpenAI(
  api_key="API-KEY"
)


# contains the aspects of each comment.
# ith element is a list containing the
# aspects of the ith comment in the dataset
aspects_list = []

# extracts a list of aspects for each comment
# and populates aspects_list
for comment in feedback_list:
    extract_aspects_prompt = f"""
    Extract the key aspects from the given student feedback comment.

    Rules:
    - Return ONLY a valid Python list of strings.
    - Each aspect MUST be a single world lemma.
    - ALL elements in the list must be quoted.

    Example output:
    ["pace", "clarity"]

    Comment: {comment}
    """

    raw_sentiment_output = client.responses.create(
        model="gpt-5.2",
        input = extract_aspects_prompt,
        store=True,
    ).output_text

    aspect_list = ast.literal_eval(raw_sentiment_output)
    aspects_list.append(aspect_list)

In [ ]:
from unsloth import FastLanguageModel
import torch
from peft import PeftModel

# connect to google colab to retrieve fine-tuned llama 3 model
from google.colab import drive
drive.mount('/content/drive')
save_dir = "/content/drive/My Drive/aspect_sentiment_classifier"

# load fine-tuned llama 3 model
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = torch.float16,
    load_in_4bit = True,
    device_map = "cuda",
)
model = PeftModel.from_pretrained(
    model,
    save_dir,
)
FastLanguageModel.for_inference(model)

# define alpaca prompt template for inference
alpaca_prompt = """### Instruction:
{}

### Input:
{}

### Response:
{}"""

Mounted at /content/drive
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
from tqdm import tqdm
import re

FastLanguageModel.for_inference(model)
model.eval()

# maps aspect to a list containing
# [total sentiment score, frequency of occurence, list of indices of comments mentioning the aspect]
aspect_score_map={}

for i in range(len(feedback_list)):
    for aspect in aspects_list[i]:
      instruction = f"""Classify the sentiment (positive, negative, or neutral) for the aspect "{aspect}" in the input sentence. Return ONLY valid JSON with schema: {{"sentiment_label": "positive|neutral|negative", "reason": "..."}}"""
      sentence = feedback_list[i]
      prompt = alpaca_prompt.format(instruction, sentence, "",)
      inputs = tokenizer([prompt], return_tensors="pt",).to(model.device)

      with torch.no_grad():
          outputs = model.generate(
              **inputs,
              max_new_tokens=128,
              use_cache=False,
              do_sample=False,
          )

      # process output
      decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
      sentiment_explanation_text = decoded.split("### Response:")[-1].strip()
      json_response = ast.literal_eval(sentiment_explanation_text)
      sentiment_label = json_response["sentiment_label"]

      sentiment_value = 0
      if sentiment_label == "positive":
        sentiment_value = 1
      elif sentiment_label == "negative":
        sentiment_value = -1
      else:
        sentiment_value = 0

      if aspect not in aspect_score_map:
        aspect_score_map[aspect] = [0,0,[]]
      aspect_score_map[aspect][0] += sentiment_value
      aspect_score_map[aspect][1] += 1
      aspect_score_map[aspect][2].append(i)

# calculate average sentiment score for each aspect
for k,v in aspect_score_map.items():
  aspect_score_map[k][0] = v[0]/v[1]

# sort
sorted_aspect_pairs_sentiment = sorted(aspect_score_map.items(), key=lambda x: x[1][0])
sorted_aspect_pairs_frequency = sorted(aspect_score_map.items(), key=lambda x: x[1][1])

if len(sorted_aspect_pairs_sentiment) < 6:
  print("There are only a few aspects covered by the submitted student feedback. Below shows the average sentiment score of each aspect where -1 indicates very negative and 1 indicates very positive.")
  print(sorted_aspect_pairs_sentiment)
else:
  top_3_sentiment_aspects = sorted_aspect_pairs_sentiment[-3:]
  bottom_3_sentiment_aspects = sorted_aspect_pairs_sentiment[:3]
  top_3_frequent_aspects = sorted_aspect_pairs_frequency[-3:]

  top_3_aspect_average_pairs = list(map(lambda x: [x[0], x[1][0]], top_3_sentiment_aspects))
  bottom_3_aspect_average_pairs = list(map(lambda x: [x[0], x[1][0]], bottom_3_sentiment_aspects))
  top_3_frequent_aspects_pairs = list(map(lambda x: [x[0], x[1][1]], top_3_frequent_aspects))

  print("The top 3 most positive rated aspects are displayed below. Below shows the average sentiment score of each aspect where -1 indicates very negative and 1 indicates very positive.")
  print(top_3_aspect_average_pairs)
  print("The bottom 3 most negative rated aspects are displayed below. Below shows the average sentiment score of each aspect where -1 indicates very negative and 1 indicates very positive.")
  print(bottom_3_aspect_average_pairs)
  print("The most frequently mentioned aspects are displayed below.")
  print(top_3_frequent_aspects_pairs)




Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=12

The top 3 most positive rated aspects are displayed below. Below shows the average sentiment score of each aspect where -1 indicates very negative and 1 indicates very positive.
[['question', 1.0], ['lesson', 1.0], ['balance', 1.0]]
The bottom 3 most negative rated aspects are displayed below. Below shows the average sentiment score of each aspect where -1 indicates very negative and 1 indicates very positive.
[['cramming', -1.0], ['repetitive', -1.0], ['instruction', -1.0]]
The most frequently mentioned aspects are displayed below.
[['lecture', 12], ['assignment', 27], ['pace', 27]]
